# Phase 2: Evaluation Framework with Ragas

**Enterprise Agentic RAG System**

In this notebook we implement and run a production-grade evaluation pipeline using **Ragas**.

Goals:
- Load the curated evaluation dataset (`data/evaluation/eval_dataset.json`)
- Run the existing `QueryEngine` (Gemma 4 8B + Chroma) on each question
- Evaluate using Ragas metrics: faithfulness, answer_relevancy, context_precision, context_recall
- Use **Gemma 4 8B** as the judge LLM and **nomic-embed-text** for embeddings (via LlamaIndex wrappers)
- Log timing and results
- Persist timestamped results to `artifacts/evaluation_results/`

This framework is fully configurable via `config.yaml` and repeatable.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys
import json
import pandas as pd
from datetime import datetime

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project root:", project_root)

## 1. Load Configuration and Configure LlamaIndex

In [ ]:
from src.config import get_settings, settings
from src.logging_config import logger

print("=== Evaluation Configuration ===")
print("Judge LLM (Ragas)   :", settings.evaluation.llm_for_judge)
print("Embeddings (Ragas)  :", settings.evaluation.embed_model_for_ragas)
print("Metrics             :", settings.evaluation.ragas_metrics)
print("Dataset path        :", settings.evaluation.dataset_path)
print("Output directory    :", settings.evaluation.output_dir)
print("Save results        :", settings.evaluation.save_results)

# Configure global LlamaIndex Settings (Gemma 4 8B + nomic-embed-text)
settings.configure_llama_index()

from llama_index.core import Settings as LlamaSettings
print("\nLlamaIndex LLM      :", LlamaSettings.llm.model)
print("LlamaIndex Embedder :", LlamaSettings.embed_model.model_name)

## 2. Load QueryEngine (from Phase 1)

In [ ]:
from src.retrieval import get_query_engine

print("Loading QueryEngine (this connects to existing Chroma vector store)...")
query_engine = get_query_engine()
print("QueryEngine ready using model:", settings.ollama.llm_model)

## 3. Preview the Evaluation Dataset

In [ ]:
dataset_path = Path(settings.paths.resolve()["evaluation"]) / "eval_dataset.json"

with open(dataset_path, "r", encoding="utf-8") as f:
    eval_dataset = json.load(f)

print(f"Total questions in dataset: {len(eval_dataset)}\n")

# Show a few examples
for i, item in enumerate(eval_dataset[:3], 1):
    print(f"--- Question {i} ---")
    print("Q:", item["question"])
    print("Ground Truth (first 200 chars):", item["ground_truth"][:200], "...")
    print()

## 4. Initialize and Run Ragas Evaluator

In [ ]:
from src.evaluation import RAGASEvaluator

evaluator = RAGASEvaluator(query_engine=query_engine)

print("Starting Ragas evaluation... (this may take several minutes with local models)")
start = datetime.now()

result = evaluator.evaluate(save=True)

print(f"\nEvaluation completed in {result.duration_seconds:.1f} seconds")

## 5. Display Results

In [ ]:
print("=== Ragas Evaluation Summary ===")
print(f"Timestamp     : {result.timestamp}")
print(f"Model         : {result.model_used}")
print(f"Judge Model   : {result.judge_model}")
print(f"# Questions   : {result.num_questions}")
print(f"Duration      : {result.duration_seconds} seconds")
print()

print("--- Metric Scores (mean) ---")
for metric, score in sorted(result.metrics.items()):
    print(f"{metric:25s}: {score:.4f}")

In [ ]:
# Nice table using pandas
scores_df = pd.DataFrame.from_dict(result.metrics, orient="index", columns=["Score"])
scores_df.index.name = "Metric"
scores_df = scores_df.sort_values("Score", ascending=False)

print("\nRagas Scores (sorted):")
display(scores_df.style.format("{:.4f}").background_gradient(cmap="RdYlGn", vmin=0, vmax=1))

## 6. Inspect Saved Artifacts

In [ ]:
output_dir = Path(settings.evaluation.output_dir)
print(f"Results saved in: {output_dir.absolute()}\n")

files = sorted(output_dir.glob(f"evaluation_{result.timestamp}*"))
for f in files:
    print(f"- {f.name}")

## 7. Load and Explore Detailed Results (if CSV was saved)

In [ ]:
import glob

detail_files = glob.glob(str(output_dir / f"evaluation_{result.timestamp}*_details.csv"))
if detail_files:
    details_df = pd.read_csv(detail_files[0])
    print("Detailed per-question scores (first 5 rows):")
    display(details_df.head().style.format(precision=3))
else:
    print("No detailed CSV found (may happen with certain Ragas versions).")

## 8. Summary & Next Steps

- ✅ Config-driven Ragas evaluation implemented
- ✅ Uses **Gemma 4 8B** as judge and **nomic-embed-text** for embeddings
- ✅ Full trajectory evaluation (answer + retrieved contexts)
- ✅ Results are logged and saved with timestamps
- ✅ Modular `RAGASEvaluator` class + convenience `run_evaluation()` function

**Typical next actions:**
- Analyze low-scoring questions and improve retrieval/chunking/prompts
- Add more metrics or custom metrics (e.g. citation accuracy)
- Run evaluation regularly as a regression test after pipeline changes
- Compare different retrieval strategies (hybrid, reranking, etc.) using the same dataset
- Integrate evaluation into CI or a simple CLI command (`rag evaluate`)

The evaluation framework is now ready for systematic improvement of the agentic RAG system.